In [3]:
from pathlib import Path
import numpy as np
import SimpleITK as sitk

from sam2.build_sam import build_sam2_video_predictor_npz
from App_utils.inference import run_medsam2_inference_from_arrays
from App_utils.prompt_utils import load_mask_like_reference
from App_utils.post_processing import (
    keep_largest_3d_connected_component,
    save_mask_like_reference,
)
from eval import compute_metrics

from unc_handling import UG_prompter


def self_prompting_pipeline(
    input_folder,
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    perform_segmentation=True,
    avg_thr_band_thickness=3.0,
    band_method="local_normal",   # "raycast" or "local_normal"
    prompt_band_threshold=0.0,
    propagation_style="default",
    keep_within_dense_buffer=True,
    dense_slice_buffer=2,
    keep_largest_component=False,
    verbose=False,
    output_folder=None,
    eval=True,
    save_report=False,
):
    """Run MEDSAM2 for all subjects using uncertainty-generated prompts via UG_prompter."""

    root = Path(input_folder)
    if not root.is_dir():
        raise FileNotFoundError(f"Input folder does not exist: {root}")

    subjects = sorted([p.name for p in root.iterdir() if p.is_dir()])
    if len(subjects) == 0:
        raise ValueError(f"No subject folders found in: {root}")

    if verbose:
        print(f"Building SAM predictor from checkpoint: {checkpoint}")
    predictor = build_sam2_video_predictor_npz(cfg, checkpoint)

    all_performance_metrics = []

    report_path = root / "pipeline_report.txt"
    if save_report:
        with open(report_path, "w") as f:
            f.write("Self-prompting pipeline report\n")
            f.write(f"Input folder: {root}\n")
            f.write(f"Volume of Interest: {volume_of_interest}\n")
            f.write("=" * 60 + "\n")

    for num, subject in enumerate(subjects, start=1):
        subject_path = root / subject / "MR_StorT2"
        img_path = subject_path / "image.nii.gz"

        if not img_path.is_file():
            if verbose:
                print(f"[{num}/{len(subjects)}] Skipping {subject}: missing image")
            continue

        if volume_of_interest == "CTVT":
            gt_path = subject_path / "mask_CTVT_427.nii.gz"
        elif volume_of_interest == "rectum":
            gt_path = subject_path / "mask_Rectum.nii.gz"
        else:
            raise ValueError("volume_of_interest must be 'CTVT' or 'rectum'")

        if verbose:
            print(f"[{num}/{len(subjects)}] Processing {subject}")

        try:
            prompter = UG_prompter(
                parentfolder=root,
                subject_nr=num - 1,
                volume_of_interest=volume_of_interest,
                verbose=verbose,
            )
        except Exception as e:
            if verbose:
                print(f"[{num}/{len(subjects)}] Skipping {subject}: {e}")
            continue

        # Use class-loaded arrays for uncertainty handling / prompting
        img = prompter.img.astype(np.float32)
        spacing = prompter.img_spacing
        dense_mask_3d = prompter.mask.astype(np.uint8)

        # Read itk image here in pipeline for saving later
        img_itk = sitk.ReadImage(str(img_path))

        # GT stays handled by pipeline
        gt_mask = load_mask_like_reference(str(gt_path), img.shape) if gt_path.is_file() else None

        if verbose and gt_mask is None:
            print(f"  Warning: GT mask not found at {gt_path}")

        try:
            prompter.threshold_uncertainty_map(
                unc_threshold=None,
                target_mm=avg_thr_band_thickness,
                step_fraction=0.02,
            )
            prompter.compute_band_thickness(method=band_method)
            generated_prompts = prompter.generate_prompts(band_threshold=prompt_band_threshold)
        except Exception as e:
            if verbose:
                print(f"[{num}/{len(subjects)}] Skipping {subject}: prompt generation failed ({e})")
            continue

        prompts_by_slice = {}

        dense_slices = np.where(
            dense_mask_3d.reshape(dense_mask_3d.shape[0], -1).sum(axis=1) > 0
        )[0].astype(int).tolist()

        generated_slices = sorted(generated_prompts.keys())
        prompted_slices = sorted(set(dense_slices + generated_slices))

        for slice_idx in prompted_slices:
            prompt_info = generated_prompts.get(slice_idx, None)

            pos_pts_slice = (
                prompt_info["positive_points"].astype(np.float32)
                if prompt_info is not None and len(prompt_info["positive_points"]) > 0
                else np.empty((0, 2), dtype=np.float32)
            )

            neg_pts_slice = (
                np.empty((0, 2), dtype=np.float32)
                if prompt_info is None or "negative_points" not in prompt_info
                else prompt_info["negative_points"].astype(np.float32)
            )

            box_slice = (
                prompt_info["boxes"][0].astype(np.float32)
                if prompt_info is not None and "boxes" in prompt_info and len(prompt_info["boxes"]) > 0
                else None
            )

            points_list = []
            labels_list = []

            if len(pos_pts_slice) > 0:
                points_list.append(pos_pts_slice)
                labels_list.append(np.ones(len(pos_pts_slice), dtype=np.int64))

            if len(neg_pts_slice) > 0:
                points_list.append(neg_pts_slice)
                labels_list.append(np.zeros(len(neg_pts_slice), dtype=np.int64))

            if len(points_list) > 0:
                points = np.concatenate(points_list, axis=0).astype(np.float32)
                point_labels = np.concatenate(labels_list, axis=0).astype(np.int64)
            else:
                points = None
                point_labels = None

            mask_input = dense_mask_3d[slice_idx].astype(np.uint8) if slice_idx in dense_slices else None

            prompts_by_slice[int(slice_idx)] = (points, point_labels, box_slice, mask_input)

        if len(prompts_by_slice) == 0:
            if verbose:
                print(f"[{num}/{len(subjects)}] Skipping {subject}: no usable prompts found")
            continue

        segs_3d = None

        if perform_segmentation:
            if verbose:
                n_pos = sum(len(v["positive_points"]) for v in generated_prompts.values())
                n_neg = sum(
                    len(v["negative_points"]) for v in generated_prompts.values()
                    if "negative_points" in v
                )
                print(
                    f"Using {len(prompted_slices)} prompted slices "
                    f"({n_pos} positive points, {n_neg} negative points)"
                )

            segs_3d = run_medsam2_inference_from_arrays(
                vol=img,
                predictor=predictor,
                image_size=image_size,
                prompts_by_slice=prompts_by_slice,
                p_low=p_low,
                p_high=p_high,
                threshold=0.0,
                propagation_style=propagation_style,
            )

            if keep_largest_component:
                segs_3d = keep_largest_3d_connected_component(segs_3d)

            if keep_within_dense_buffer:
                dense_slices_arr = np.asarray(dense_slices, dtype=int)
                if len(dense_slices_arr) > 0:
                    keep_mask = np.zeros_like(segs_3d, dtype=np.uint8)
                    max_z = segs_3d.shape[0] - 1
                    for z0 in dense_slices_arr:
                        z_start = max(0, int(z0) - int(dense_slice_buffer))
                        z_end = min(max_z, int(z0) + int(dense_slice_buffer))
                        keep_mask[z_start:z_end + 1] = 1
                    segs_3d = (segs_3d > 0).astype(np.uint8) * keep_mask

            if output_folder is None:
                out_dir = subject_path / "SAM_OUTPUT"
            else:
                out_dir = Path(output_folder) / subject
            out_dir.mkdir(parents=True, exist_ok=True)

            out_path = out_dir / f"pred_{subject}_{volume_of_interest}.nii.gz"
            save_mask_like_reference(segs_3d, img_itk, str(out_path))

            if verbose:
                print(f"  Saved: {out_path}")

        if eval and segs_3d is not None and gt_mask is not None:
            performance_metrics = compute_metrics(
                segs_3d,
                gt_mask,
                spacing=spacing,
                surface_dice_tol=2.0,
                apl_tol=0.0,
            )
            all_performance_metrics.append(performance_metrics)

            if save_report:
                with open(report_path, "a") as f:
                    f.write("-" * 60 + "\n")
                    f.write(f"Subject: {subject}\n")
                    f.write(f"Volume of Interest: {volume_of_interest}\n")
                    f.write(f"Number of Prompted Slices: {len(prompts_by_slice)}\n")
                    f.write(f"Number of Positive Prompts: {sum(len(v['positive_points']) for v in generated_prompts.values())}\n")
                    f.write(
                        f"Number of Negative Prompts: "
                        f"{sum(len(v['negative_points']) for v in generated_prompts.values() if 'negative_points' in v)}\n"
                    )
                    f.write("Performance Metrics:\n")
                    for metric_name, metric_value in performance_metrics.items():
                        f.write(f"  {metric_name}: {metric_value:.4f}\n")
                    f.write("\n")

    return all_performance_metrics

In [4]:
all_performance_metrics = self_prompting_pipeline(
    input_folder = "data\LUNDPROBE\ExtendedSamples\development",
    volume_of_interest="CTVT",
    checkpoint="checkpoints/MedSAM2_latest.pt",
    cfg="configs/sam2.1_hiera_t512.yaml",
    image_size=512,
    p_low=1.0,
    p_high=99.0,
    perform_segmentation=True,
    avg_thr_band_thickness=3.0,
    band_method="local_normal",   # "raycast" or "local_normal"
    prompt_band_threshold=3.0,    # only prompt slices above this slice-level band thickness
    propagation_style="default",
    keep_within_dense_buffer=True,
    dense_slice_buffer=2,
    keep_largest_component=False,
    verbose=False,
    output_folder=None,
    eval=True,
    save_report=True,
)

<>:2: SyntaxWarning: invalid escape sequence '\L'
<>:2: SyntaxWarning: invalid escape sequence '\L'
C:\Users\20202310\AppData\Local\Temp\ipykernel_18484\53649314.py:2: SyntaxWarning: invalid escape sequence '\L'
  input_folder = "data\LUNDPROBE\ExtendedSamples\development",


[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(12.057692307692308), np.float64(7.436619718309859), np.float64(5.310326086956523), np.float64(3.596634615384615), np.float64(3.0166666666666666), np.float64(2.525099601593625), np.float64(2.152346570397112), np.float64(1.8615646258503404), np.float64(1.8684210526315794), np.float64(1.84640522875817), np.float64(1.7721854304635762), np.float64(1.8820338983050848), np.float64(2.0063157894736845), np.float64(2.5163568773234206), np.float64(3.168016194331984), np.float64(3.053953488372094), np.float64(3.274175824175824), np.float64(4.94469696969697), 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [27, 2

c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (default)...


propagate in video: 100%|██████████| 61/61 [00:13<00:00,  4.68it/s]


...

Saved mask to: data\LUNDPROBE\ExtendedSamples\development\newAcq_f7b01268d65999c6\MR_StorT2\SAM_OUTPUT\pred_newAcq_f7b01268d65999c6_CTVT.nii.gz
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, np.float64(7.12542372881356), np.float64(9.094999999999999), np.float64(4.650310559006211), np.float64(3.292045454545455), np.float64(3.1862433862433863), np.float64(3.0684466019417473), np.float64(2.404504504504504), np.float64(1.8229787234042552), np.float64(1.8216326530612243), np.float64(2.4764940239043827), np.float64(3.146031746031746), np.float64(4.157203389830509), np.float64(2.526486486486487), np.float64(2.2037267080745346), np.float64(2.912698412698413), np.float64(6.893181818181819), 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

propagate in video: 100%|██████████| 57/57 [00:12<00:00,  4.74it/s]


Backward propagation (default)...


propagate in video: 100%|██████████| 32/32 [00:08<00:00,  3.60it/s]


Saved mask to: data\LUNDPROBE\ExtendedSamples\development\newAcq_fa53f60ea516d2da\MR_StorT2\SAM_OUTPUT\pred_newAcq_fa53f60ea516d2da_CTVT.nii.gz
